In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
sentence-transformers \
transformers \
accelerate \
bitsandbytes \
peft \
pypdf \
pymupdf \
networkx \
lancedb \
pyarrow \
pandas \
numpy \
tqdm \
scikit-learn \
datasets \
ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.9/369.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ROOT_FOLDER = "/content/drive/MyDrive/Data (RAG)"

In [5]:
# ================================================================
# CONFIG — Centralized constants for the TIET RAG pipeline
# ================================================================
import logging
import sys

# --- Logging ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
log = logging.getLogger("tiet_rag")

# --- Paths ---
CHECKPOINT_DIR = "/content/drive/MyDrive/tiet_rag/knowledge_graph"
QA_DATASET_DIR = "/content/drive/MyDrive/tiet_rag/qa_dataset"

# --- Model params ---
LLM_TEMP_EXTRACTION = 0.15
LLM_TEMP_QA = 0.3
LLM_MAX_TOKENS = 1200
LLM_MAX_TOKENS_SHORT = 600
LLM_REPETITION_PENALTY = 1.05

# --- Embedding ---
EMBEDDER_MODEL = "BAAI/bge-m3"
EMBEDDING_BATCH_SIZE = 16

# --- Graph building ---
MIN_CHUNK_LENGTH = 100
ENTITY_MIN_NAME_LEN = 3
SIM_LOW = 0.45
SIM_HIGH = 0.85
MAX_LONG_DISTANCE_PAIRS = 150
LONG_DISTANCE_SAMPLE_RATE = 3
LONG_DISTANCE_TOP_K = 2
LD_PROMPT_MAX_TEXT = 700

# --- QA generation ---
QA_CHUNK_TEXT_LIMIT = 500
PROPOSALS_PER_CHUNK = 3
MAX_QA_PAIRS_PER_CHUNK = 2
MAX_RETRIES = 2
GROUNDEDNESS_SIM_THRESHOLD = 0.72
DEDUP_SIM_THRESHOLD = 0.92


In [6]:
import os

print(os.path.exists(ROOT_FOLDER))

True


In [ ]:
# ================================================================
# Fix 4: RESUME — Load chunk_registry from Drive if available
# If chunk_registry_full.json exists, skip Cells 7-11 entirely.
# Run this cell at the start of every subsequent session.
# ================================================================

import json
import os

CHUNK_REGISTRY_PATH = f"{CHECKPOINT_DIR}/chunk_registry_full.json"

chunk_registry = None
if os.path.exists(CHUNK_REGISTRY_PATH):
    with open(CHUNK_REGISTRY_PATH) as f:
        chunk_registry = json.load(f)
    log.info("Loaded chunk registry from disk: %d chunks", len(chunk_registry))
    print(f"✓ Loaded {len(chunk_registry)} chunks from Drive — skip Cells 7-11")
else:
    print("No saved chunk_registry found — run Cells 7-11 for first-time ingestion")


In [7]:
pdf_files = []

for root, dirs, files in os.walk(ROOT_FOLDER):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(
                os.path.join(root, file)
            )

print("Total PDFs:", len(pdf_files))

Total PDFs: 2


In [8]:
# ================================================================
# CELL: PDF LOADING + TEXT CLEANING (Fix 1 — structure-aware)
# Uses fitz (PyMuPDF) for page-level text extraction with cleaning
# ================================================================

import fitz
import re
import os
import hashlib
from tqdm import tqdm

# ── Noise patterns to strip ──
_GARBAGE_RE = re.compile(
    r'^(Page\s+\d+(\s+of\s+\d+)?|TIET[,\s]*PATIALA|Thapar Institute.*|'
    r'\d+\s*$|Version:?\s*[\d.]+|Release\s+(No|Date).*|'
    r'PREPARED BY.*|APPROVED BY.*|Confidential.*|'
    r'All\s+Rights\s+Reserved.*|www\.thapar\.edu.*)',
    re.IGNORECASE
)

_WHITESPACE_RE = re.compile(r'[ \t]+')
_BLANK_LINES_RE = re.compile(r'\n{3,}')


def clean_pdf_text(text: str) -> str:
    """
    Clean raw PDF page text:
    - Remove headers, footers, boilerplate
    - Normalize whitespace
    - Merge broken lines inside paragraphs
    - Preserve bullet lists and numbered clauses
    """
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        stripped = line.strip()

        # Skip garbage lines
        if _GARBAGE_RE.match(stripped):
            continue

        # Skip very short noise (single chars, dashes-only, etc.)
        if stripped and len(stripped) <= 2 and not stripped[0].isdigit():
            continue

        # Normalize internal whitespace
        stripped = _WHITESPACE_RE.sub(" ", stripped)

        cleaned.append(stripped)

    # Join lines, then merge broken lines inside paragraphs
    result = "\n".join(cleaned)

    # Merge lines that are broken mid-sentence:
    # A line ending with a lowercase letter followed by a line starting
    # with a lowercase letter indicates a broken line
    result = re.sub(
        r'([a-z,;])\n([a-z])',
        r'\1 \2',
        result
    )

    # Collapse excessive blank lines
    result = _BLANK_LINES_RE.sub("\n\n", result)

    return result.strip()


# ── Load PDFs page-by-page ──
documents = []

for pdf_path in tqdm(pdf_files, desc="Loading PDFs"):
    try:
        doc = fitz.open(pdf_path)
        pages = []
        for page_num in range(len(doc)):
            raw_text = doc[page_num].get_text("text")
            cleaned = clean_pdf_text(raw_text)
            if cleaned.strip():
                pages.append({
                    "page_num": page_num + 1,  # 1-indexed
                    "text": cleaned,
                })
        doc.close()

        if pages:
            documents.append({
                "document_name": os.path.basename(pdf_path),
                "pdf_path": pdf_path,
                "pages": pages,
                "total_pages": len(pages),
            })

    except Exception as e:
        print(f"Error loading {pdf_path}: {e}")

print(f"Documents Loaded: {len(documents)}")
print(f"Total pages: {sum(d['total_pages'] for d in documents)}")

Loading PDFs: 100%|██████████| 2/2 [00:03<00:00,  1.67s/it]

Documents Loaded: 2
Total pages: 7


In [9]:
# ================================================================
# CELL: SECTION / HEADING DETECTION (before chunking)
# Detects headings and assigns body text to sections
# ================================================================

import re

# ── Heading detection heuristics ──
_HEADING_NUMBERED = re.compile(
    r'^(\d+\.)+\d*\s+[A-Z]'  # e.g. 1.1 Title, 2.3.4 Something
)
_HEADING_KEYWORD = re.compile(
    r'^(Section|Clause|Rule|Article|Policy|Procedure|Eligibility|'
    r'Exception|Deadline|Schedule|Appendix|Annexure|Part)\s+',
    re.IGNORECASE
)
_HEADING_ALLCAPS = re.compile(
    r'^[A-Z][A-Z\s\-&/,]{4,80}$'  # ALL CAPS lines, 5-80 chars
)
_HEADING_TITLE_CASE = re.compile(
    r'^[A-Z][a-z]+(\s+[A-Za-z]+){1,8}$'  # Short title-cased line
)


def is_heading(line: str) -> bool:
    """Check if a line looks like a section heading."""
    line = line.strip()
    if not line or len(line) > 120:
        return False
    if len(line) < 3:
        return False

    # Numbered heading: 1. Title, 1.1 Title, etc.
    if _HEADING_NUMBERED.match(line):
        return True

    # Keyword heading: Section 1, Clause 2, Rule 3, etc.
    if _HEADING_KEYWORD.match(line):
        return True

    # ALL CAPS heading (but not too short or too long)
    if _HEADING_ALLCAPS.match(line) and len(line) >= 5:
        # Avoid false positives on abbreviations
        words = line.split()
        if len(words) >= 2:
            return True

    # Short title-cased standalone line (likely a subsection)
    if _HEADING_TITLE_CASE.match(line) and len(line) <= 60:
        word_count = len(line.split())
        if 2 <= word_count <= 8:
            return True

    return False


def extract_document_structure(doc: dict) -> list:
    """
    Extract structured segments from a document's pages.

    Input: doc dict with 'pages' list (each page has 'page_num' and 'text')
    Output: list of segment dicts:
        - document_name
        - page_start, page_end
        - section_title
        - subsection_title
        - body_text
    """
    segments = []
    current_section = "General"
    current_subsection = ""
    current_body_lines = []
    current_page_start = 1
    current_page_end = 1

    for page in doc["pages"]:
        page_num = page["page_num"]
        lines = page["text"].split("\n")

        for line in lines:
            stripped = line.strip()
            if not stripped:
                if current_body_lines:
                    current_body_lines.append("")
                continue

            if is_heading(stripped):
                # Save previous segment if it has content
                body = "\n".join(current_body_lines).strip()
                if body and len(body) >= 20:
                    segments.append({
                        "document_name": doc["document_name"],
                        "page_start": current_page_start,
                        "page_end": current_page_end,
                        "section_title": current_section,
                        "subsection_title": current_subsection,
                        "body_text": body,
                    })

                # Determine if this is a section or subsection
                if _HEADING_NUMBERED.match(stripped):
                    # Count dots to determine depth
                    num_part = stripped.split()[0]
                    depth = num_part.count(".")
                    if depth <= 1:
                        current_section = stripped[:100]
                        current_subsection = ""
                    else:
                        current_subsection = stripped[:100]
                elif _HEADING_ALLCAPS.match(stripped):
                    current_section = stripped[:100]
                    current_subsection = ""
                else:
                    current_subsection = stripped[:100]

                current_body_lines = []
                current_page_start = page_num
                current_page_end = page_num
            else:
                current_body_lines.append(stripped)
                current_page_end = page_num

    # Don't forget the last segment
    body = "\n".join(current_body_lines).strip()
    if body and len(body) >= 20:
        segments.append({
            "document_name": doc["document_name"],
            "page_start": current_page_start,
            "page_end": current_page_end,
            "section_title": current_section,
            "subsection_title": current_subsection,
            "body_text": body,
        })

    return segments


# ── Run structure extraction on all documents ──
all_segments = []
for doc in tqdm(documents, desc="Extracting structure"):
    segs = extract_document_structure(doc)
    all_segments.extend(segs)

print(f"Total structured sections: {len(all_segments)}")
print(f"\nSample sections:")
for seg in all_segments[:5]:
    print(f"  [{seg['document_name']}] p{seg['page_start']}-{seg['page_end']}")
    print(f"    Section: {seg['section_title']}")
    print(f"    Subsection: {seg['subsection_title']}")
    print(f"    Body length: {len(seg['body_text'])} chars")
    print()

Extracting structure: 100%|██████████| 2/2 [00:00<00:00, 1456.10it/s]

Total structured sections: 13

Sample sections:
  [ADMISSION TO ME-MTECH PROGRAMS.pdf] p1-1
    Section: General
    Subsection: 
    Body length: 84 chars

  [ADMISSION TO ME-MTECH PROGRAMS.pdf] p1-1
    Section: ELIGIBILITY FOR ADMISSION
    Subsection: 
    Body length: 1516 chars

  [ADMISSION TO ME-MTECH PROGRAMS.pdf] p1-2
    Section: ELIGIBILITY FOR ADMISSION
    Subsection: Thermal Engineering
    Body length: 3038 chars

  [ADMISSION TO ME-MTECH PROGRAMS.pdf] p2-2
    Section: DURATION OF PROGRAMME
    Subsection: Regular Mode
    Body length: 217 chars

  [ADMISSION TO ME-MTECH PROGRAMS.pdf] p2-3
    Section: DURATION OF PROGRAMME
    Subsection: Flexible Mode
    Body length: 275 chars



In [ ]:
# ================================================================
# CELL: SEMANTIC CHUNKING + CHUNK REGISTRY BUILDER
# Structure-aware chunking that respects section boundaries
# ================================================================

import hashlib
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Bullet / numbered clause boundary detection ──
_BULLET_RE = re.compile(
    r'^(\s*[-•▪▸►*]\s+|\s*\(?[a-z0-9]+[.)]\s+|\s*\d+\.\s+)',
)


def semantic_chunk_section(
    section_text: str,
    max_chars: int = 1200,
    overlap: int = 150,
) -> list:
    """
    Split section text into semantic chunks.

    Priority:
    1. Split on paragraph breaks (\n\n)
    2. Split on bullet / numbered clause boundaries
    3. Fall back to RecursiveCharacterTextSplitter if no structure

    Rules:
    - Target 800-1200 chars per chunk
    - Keep bullet lists together when possible
    - Isolate exception/eligibility/deadline clauses if self-contained
    """
    if len(section_text) <= max_chars:
        return [section_text]

    # ── Strategy 1: Split on paragraph breaks ──
    paragraphs = re.split(r'\n\n+', section_text)

    # If only one big paragraph, try bullet splits
    if len(paragraphs) <= 1:
        # Try splitting on bullet points
        lines = section_text.split("\n")
        bullet_groups = []
        current_group = []

        for line in lines:
            if _BULLET_RE.match(line) and current_group:
                # Start of new bullet — save current group if big enough
                group_text = "\n".join(current_group)
                if len(group_text) > max_chars:
                    # Group is too big, will fall back later
                    bullet_groups.append(group_text)
                    current_group = [line]
                else:
                    current_group.append(line)
            else:
                current_group.append(line)

        if current_group:
            bullet_groups.append("\n".join(current_group))

        if len(bullet_groups) > 1:
            paragraphs = bullet_groups

    # ── Merge small paragraphs, split large ones ──
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # If adding this paragraph would exceed max, save current and start new
        if current_chunk and len(current_chunk) + len(para) + 2 > max_chars:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            if current_chunk:
                current_chunk += "\n\n" + para
            else:
                current_chunk = para

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    # ── Handle any chunk that's still too large: fall back to splitter ──
    final_chunks = []
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_chars,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " "],
    )

    for chunk in chunks:
        if len(chunk) <= max_chars:
            final_chunks.append(chunk)
        else:
            sub_chunks = fallback_splitter.split_text(chunk)
            final_chunks.extend(sub_chunks)

    # ── Add overlap between chunks ──
    if len(final_chunks) > 1 and overlap > 0:
        overlapped = [final_chunks[0]]
        for i in range(1, len(final_chunks)):
            prev_tail = final_chunks[i - 1][-overlap:]
            # Only add overlap if it doesn't start mid-word
            space_idx = prev_tail.find(" ")
            if space_idx >= 0:
                prev_tail = prev_tail[space_idx + 1:]
            overlapped.append(prev_tail + "\n" + final_chunks[i])
        final_chunks = overlapped

    return final_chunks


def build_chunk_registry(documents: list, all_segments: list) -> list:
    """
    Build the final chunk registry with full provenance metadata.

    Each chunk contains:
    - chunk_id, document_name, pdf_path
    - page_start, page_end
    - section_title, subsection_title
    - chunk_text, char_count, source_text_hash
    - category
    """
    registry = []

    # Build a map from document_name to pdf_path and category
    doc_info = {}
    for doc in documents:
        category = doc["pdf_path"].split("/")[-2].lower() if "/" in doc["pdf_path"] else "general"
        # Also try backslash for Windows paths
        if category == "general" and "\\" in doc["pdf_path"]:
            parts = doc["pdf_path"].split("\\")
            category = parts[-2].lower() if len(parts) >= 2 else "general"
        doc_info[doc["document_name"]] = {
            "pdf_path": doc["pdf_path"],
            "category": category,
        }

    for segment in all_segments:
        doc_name = segment["document_name"]
        info = doc_info.get(doc_name, {"pdf_path": "", "category": "general"})

        # Chunk the segment body text
        chunks = semantic_chunk_section(
            segment["body_text"],
            max_chars=1200,
            overlap=150,
        )

        for chunk_text in chunks:
            if len(chunk_text.strip()) < 30:
                continue

            text_hash = hashlib.sha256(
                chunk_text.encode("utf-8")
            ).hexdigest()[:16]

            registry.append({
                "chunk_id": f"{doc_name.replace('.pdf','').replace(' ','_')}__{text_hash[:12]}",
                "document_name": doc_name,
                "pdf_path": info["pdf_path"],
                "page_start": segment["page_start"],
                "page_end": segment["page_end"],
                "section_title": segment["section_title"],
                "subsection_title": segment["subsection_title"],
                "chunk_text": chunk_text,
                "char_count": len(chunk_text),
                "source_text_hash": text_hash,
                "category": info["category"],
            })


    return registry


# ── Build chunk registry ──
chunk_registry = build_chunk_registry(documents, all_segments)
print(f"Total Chunks: {len(chunk_registry)}")
# Fix 4: Persist chunk_registry to Drive (run once, skip ingestion on resume)
CHUNK_REGISTRY_PATH = f"{CHECKPOINT_DIR}/chunk_registry_full.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(CHUNK_REGISTRY_PATH):
    with open(CHUNK_REGISTRY_PATH, "w", encoding="utf-8") as f:
        json.dump(chunk_registry, f, ensure_ascii=False)
    log.info("Saved full chunk registry: %d chunks", len(chunk_registry))
else:
    log.info("chunk_registry_full.json already exists on Drive")


In [11]:
print(f"Total chunks: {len(chunk_registry)}")
if len(chunk_registry) > 0:
    print(f"Chunk keys: {chunk_registry[0].keys()}")
else:
    print("No chunks were generated. Please verify that your ROOT_FOLDER contains valid PDF files and they are being loaded correctly.")

Total chunks: 27
Chunk keys: dict_keys(['chunk_id', 'document_name', 'pdf_path', 'page_start', 'page_end', 'section_title', 'subsection_title', 'chunk_text', 'char_count', 'source_text_hash', 'category'])


In [12]:
import json
import os

out_dir = "./parsed_chunks"
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "chunks_v2.json")

if len(chunk_registry) > 0:
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(chunk_registry, f, indent=2, ensure_ascii=False)
    print(f"Chunks Saved to {out_file}")
else:
    print("No chunks to save.")


Chunks Saved to ./parsed_chunks/chunks_v2.json


In [13]:
# ================================================================
# VALIDATION — Fix 1 checklist
# ================================================================

print("=" * 60)
print("  INGESTION PIPELINE VALIDATION")
print("=" * 60)

# 1. Total documents
print(f"\n  1. Total documents:          {len(documents)}")

# 2. Total pages
total_pages = sum(d['total_pages'] for d in documents)
print(f"  2. Total pages:              {total_pages}")

# 3. Total structured sections
print(f"  3. Total structured sections:{len(all_segments)}")

# 4. Total chunks
print(f"  4. Total chunks:             {len(chunk_registry)}")

# 5. Chunk size statistics
char_counts = [c['char_count'] for c in chunk_registry]
print(f"\n  Chunk size stats:")
if char_counts:
    print(f"    Min:    {min(char_counts)} chars")
    print(f"    Max:    {max(char_counts)} chars")
    print(f"    Mean:   {sum(char_counts) / len(char_counts):.0f} chars")
    print(f"    Median: {sorted(char_counts)[len(char_counts)//2]} chars")
else:
    print("    No chunks available for statistics.")

# 6. 10 sample chunks
print(f"\n{'=' * 60}")
print("  SAMPLE CHUNKS (first 10)")
print("=" * 60)

for chunk in chunk_registry[:10]:
    print(f"\n  chunk_id:       {chunk['chunk_id']}")
    print(f"  document_name:  {chunk['document_name']}")
    print(f"  pages:          {chunk['page_start']}-{chunk['page_end']}")
    print(f"  section_title:  {chunk['section_title']}")
    print(f"  subsection:     {chunk['subsection_title']}")
    print(f"  char_count:     {chunk['char_count']}")
    print(f"  text_hash:      {chunk['source_text_hash']}")
    print(f"  text preview:   {chunk['chunk_text'][:300]}...")
    print(f"  {'─' * 50}")

print(f"\n{'=' * 60}")
print("  Validation complete. Manually check:")
print("  - Chunks stay inside one section")
print("  - Noisy headers/footers are gone")
print("  - Short rules and exceptions are preserved")
print("=" * 60)

  INGESTION PIPELINE VALIDATION

  1. Total documents:          2
  2. Total pages:              7
  3. Total structured sections:13
  4. Total chunks:             27

  Chunk size stats:
    Min:    84 chars
    Max:    1307 chars
    Mean:   661 chars
    Median: 627 chars

  SAMPLE CHUNKS (first 10)

  chunk_id:       chunk_00000
  document_name:  ADMISSION TO ME-MTECH PROGRAMS.pdf
  pages:          1-1
  section_title:  General
  subsection:     
  char_count:     84
  text_hash:      587a25b4fb7e2966
  text preview:   ADMISSION TO M.E./M.Tech. PROGRAMME (2026-27)

Mode of Program: Regular/Flexible
3.1...
  ──────────────────────────────────────────────────

  chunk_id:       chunk_00001
  document_name:  ADMISSION TO ME-MTECH PROGRAMS.pdf
  pages:          1-1
  section_title:  ELIGIBILITY FOR ADMISSION
  subsection:     
  char_count:     1029
  text_hash:      4286a0dbe60c68e6
  text preview:   Admission to all the M.E./M.Tech. programmes shall be made on the basis of valid GATE

In [14]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig

import torch

In [ ]:
# Fix 3: Use Drive-cached model if available, else download from HF
MODEL_CACHE_DIR = "/content/drive/MyDrive/tiet_rag/model_cache/qwen2.5-7b"

model_name = MODEL_CACHE_DIR if os.path.exists(MODEL_CACHE_DIR) else "Qwen/Qwen2.5-7B-Instruct"
log.info("Model source: %s", model_name)

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(

    model_name,

    quantization_config=bnb_config,

    device_map="auto"
)

print("Qwen Loaded")
# GPU Memory Check
import torch
if torch.cuda.is_available():
    print(f'Allocated VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
    print(f'Reserved VRAM:  {torch.cuda.memory_reserved() / 1e9:.2f} GB')

# Fix 3: Cache model weights to Drive after first download
if not os.path.exists(MODEL_CACHE_DIR):
    os.makedirs(os.path.dirname(MODEL_CACHE_DIR), exist_ok=True)
    tokenizer.save_pretrained(MODEL_CACHE_DIR)
    model.save_pretrained(MODEL_CACHE_DIR)
    log.info("Model cached to Drive at %s", MODEL_CACHE_DIR)


In [17]:
print(type(tokenizer))
print(type(model))

<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>


In [18]:
NODE_TYPES = [
    "POLICY",
    "RULE",
    "PROCESS",
    "ELIGIBILITY",
    "CONDITION",
    "EXCEPTION",
    "DEADLINE",
    "COMMITTEE",
    "ROLE",
    "AUTHORITY",
    "FORM",
    "PORTAL",
    "SERVICE",
    "DOCUMENT",
    "STUDENT_GROUP",
    "CONSEQUENCE"
]

EDGE_TYPES = [
    "DEPENDS_ON",
    "REQUIRES",
    "RESULTS_IN",
    "CONTAINS",
    "IMPLEMENTED_BY",
    "EXCEPTION_TO",
    "ELIGIBLE_FOR",
    "APPROVED_BY",
    "APPLIES_TO",
    "USES",
    "MANAGES",
    "OVERSEES",
    "REPORTS_TO",
    "GOVERNED_BY",
    "RELATED_TO"
]

In [19]:
# ================================================================
# NOTE: The old post-hoc section detection (build_section_index /
# get_section_for_chunk) has been REMOVED as part of Fix 1.
# Section detection now happens BEFORE chunking in the new pipeline.
# ================================================================
# (This cell intentionally left as a note — no code to run)

In [20]:
# ================================================================
# NOTE: The old section enrichment loop has been REMOVED as part
# of Fix 1. Chunk metadata is now created correctly during the
# chunking step itself — no post-hoc patching needed.
# ================================================================
# (This cell intentionally left as a note — no code to run)

In [21]:
#================================================================
# CELL B — LLM HELPER + STRICT SCHEMA VALIDATORS (Fix 2)
# Cleaner, handles errors, adds temperature control
# + strict validation for every LLM output
# ================================================================

import torch
import gc
import logging
import re
import json

log = logging.getLogger("tiet_rag")

def llm_call(prompt: str, max_tokens: int = 1200, temp: float = 0.15) -> str:
    """
    Single LLM call with proper error handling.
    temp=0.15 for extraction (factual), 0.3 for QA generation (creative)
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,          # greedy — faster and more deterministic JSON output
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=LLM_REPETITION_PENALTY,
        )

    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    del inputs, out
    gc.collect()
    return response.strip()


def llm_call_with_retry(prompt: str, max_tokens: int = 1200, temp: float = 0.15, retries: int = 2) -> str:
    """
    Call LLM with retry logic for transient failures.
    Returns empty string if all retries fail.
    """
    for attempt in range(1, retries + 2):
        try:
            return llm_call(prompt, max_tokens, temp)
        except Exception as e:
            log.warning("LLM call attempt %d/%d failed: %s", attempt, retries + 1, e)
            if attempt > retries:
                log.error("All LLM retries exhausted")
                return ""
    return ""


def safe_json_parse(text: str) -> dict | list | None:
    """Robust JSON extraction from LLM output."""
    # Strip markdown fences
    text = re.sub(r'```(?:json)?', '', text).strip()

    # Try full parse first
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try finding the outermost array
    start = text.find('[')
    end = text.rfind(']')
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass

    # Try finding the outermost object
    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass

    return None


# ================================================================
# STRICT SCHEMA VALIDATORS (Fix 2)
# ================================================================

VALID_NODE_TYPES = set(NODE_TYPES)
VALID_EDGE_TYPES = set(EDGE_TYPES)


def validate_entity(entity: dict) -> bool:
    """
    Validate a single entity dict.
    Required: name (str, non-empty), type (str, in NODE_TYPES), description (str, non-empty).
    """
    if not isinstance(entity, dict):
        return False

    name = entity.get("name")
    etype = entity.get("type")
    desc = entity.get("description")

    if not isinstance(name, str) or not name.strip():
        return False
    if not isinstance(etype, str) or etype not in VALID_NODE_TYPES:
        return False
    if not isinstance(desc, str) or not desc.strip():
        return False

    return True


def validate_relationship(rel: dict, valid_entity_names: set = None) -> bool:
    """
    Validate a single relationship dict.
    Required: source (str), relation (str, in EDGE_TYPES), target (str), description (str).
    source != target. If valid_entity_names is given, both endpoints must be in it.
    """
    if not isinstance(rel, dict):
        return False

    source = rel.get("source")
    target = rel.get("target")
    relation = rel.get("relation")
    desc = rel.get("description")

    if not isinstance(source, str) or not source.strip():
        return False
    if not isinstance(target, str) or not target.strip():
        return False
    if not isinstance(relation, str) or relation not in VALID_EDGE_TYPES:
        return False
    if not isinstance(desc, str):
        return False

    # source and target cannot be identical
    if normalize_name(source) == normalize_name(target):
        return False

    # If we have a set of valid entity names, check membership
    if valid_entity_names is not None:
        if normalize_name(source) not in valid_entity_names:
            return False

    return True


def validate_qa_structure(qa: dict) -> bool:
    """Fix 4: Strict structure validation for QA items."""
    if not isinstance(qa, dict): return False

    for key in ["question", "answer", "question_type", "difficulty"]:
        val = qa.get(key)
        if not isinstance(val, str) or not val.strip():
            return False

    # Optional metadata checks
    if "hop_count" in qa and not isinstance(qa["hop_count"], (int, float)): return False
    if "grounded" in qa and not isinstance(qa["grounded"], bool): return False
    if "source_chunks" in qa and not isinstance(qa["source_chunks"], list): return False

    return True

def validate_hop_consistency(qa: dict, expected_hop_type: str) -> bool:
    """Fix 4: Ensure multi-hop items contain necessary reasoning chains and evidence."""
    if expected_hop_type == "single_hop":
        # Single hop should have an evidence field if prompted, or at least a simple answer.
        # It should NOT have a reasoning_chain.
        pass
    else:
        # Cross-chunk and multi-hop MUST have a reasoning chain
        if "reasoning_chain" not in qa or not isinstance(qa["reasoning_chain"], list):
            return False
        if len(qa["reasoning_chain"]) < 2:
            return False

    return True

def validate_qa_item(qa: dict, expected_hop_type: str = None) -> bool:
    """Wrapper to apply both structure and consistency checks."""
    if not validate_qa_structure(qa):
        return False
    if expected_hop_type and not validate_hop_consistency(qa, expected_hop_type):
        return False
    return True


def validate_and_filter_extraction(extraction: dict) -> dict:
    """
    Strictly validate an entity+relationship extraction result.
    - Filter out invalid entities
    - Filter out invalid relationships (must reference valid entity names)
    - Log what was dropped
    Returns cleaned dict with 'entities' and 'relationships' lists.
    """
    raw_entities = extraction.get("entities", [])
    raw_relations = extraction.get("relationships", [])

    if not isinstance(raw_entities, list):
        raw_entities = []
    if not isinstance(raw_relations, list):
        raw_relations = []

    # Validate entities
    valid_entities = []
    dropped_entities = 0
    for e in raw_entities:
        if validate_entity(e):
            valid_entities.append(e)
        else:
            dropped_entities += 1

    # Build valid name set for relationship validation
    valid_names = {normalize_name(e["name"]) for e in valid_entities}

    # Validate relationships
    valid_relations = []
    dropped_relations = 0
    for r in raw_relations:
        if validate_relationship(r, valid_names):
            valid_relations.append(r)
        else:
            dropped_relations += 1

    total_dropped = dropped_entities + dropped_relations
    if total_dropped > 0:
        log.info(
            "  [validate] dropped %d entities, %d relations (invalid schema/types)",
            dropped_entities, dropped_relations
        )

    return {"entities": valid_entities, "relationships": valid_relations}


def validate_and_filter_qa_list(qa_list: list, expected_hop_type: str = None) -> list:
    """
    Validate a list of QA items. Returns only valid ones.
    Logs what was dropped.
    """
    if not isinstance(qa_list, list):
        log.warning("  [validate_qa] expected list, got %s — discarding", type(qa_list).__name__)
        return []

    valid = []
    dropped = 0
    for qa in qa_list:
        if validate_qa_item(qa, expected_hop_type):
            valid.append(qa)
        else:
            dropped += 1

    if dropped > 0:
        log.info("  [validate_qa] dropped %d/%d QA items (invalid schema)", dropped, len(qa_list))

    return valid


log.info("LLM helpers + schema validators loaded (Fix 2)")

In [22]:
#================================================================
# CELL C — HIGH-QUALITY ENTITY + RELATIONSHIP EXTRACTION (Fix 2)
# Key upgrades from Fix 2:
#   - Strict schema validation before graph insertion
#   - Retry on validation failure
#   - Stricter prompt format instructions
# ================================================================

ENTITY_EXTRACTION_PROMPT = """You are an expert knowledge graph builder for TIET (Thapar Institute of Engineering & Technology) institutional documents.

DOCUMENT: {doc_name}
CATEGORY: {category}
SECTION: {section}

TEXT:
\"\"\"
{chunk_text}
\"\"\"

TASK: Extract ALL meaningful entities AND relationships from this policy text.

ENTITY TYPES (use exactly these):
POLICY, RULE, PROCESS, ELIGIBILITY, CONDITION, EXCEPTION, DEADLINE,
COMMITTEE, ROLE, AUTHORITY, FORM, PORTAL, SERVICE, DOCUMENT, CONSEQUENCE

RELATIONSHIP TYPES (use exactly these):
DEPENDS_ON, REQUIRES, RESULTS_IN, CONTAINS, IMPLEMENTED_BY,
EXCEPTION_TO, ELIGIBLE_FOR, APPROVED_BY, APPLIES_TO, GOVERNED_BY, RELATED_TO

CRITICAL RULES:
- entity "name" must be a SHORT specific label (3-8 words max)
- entity "type" must be one of the ENTITY TYPES listed above
- entity "description" must be a complete sentence explaining what it means IN CONTEXT
- For relationships: source and target must exactly match entity names you defined
- relation must be one of the RELATIONSHIP TYPES listed above
- Extract ALL entities — don't miss important ones
- Named entities (committees, forms, specific rules) are most important

Output ONLY valid JSON. Do not include markdown. Use exactly the fields specified.
Wrong keys or extra text will be rejected. Return an empty list if nothing valid exists.

{{
  "entities": [
    {{
      "name": "Short Entity Name",
      "type": "ENTITY_TYPE",
      "description": "Full sentence describing what this entity is in context of {doc_name}.",
      "source_relation": "DEFINE"
    }}
  ],
  "relationships": [
    {{
      "source": "Entity Name A",
      "relation": "RELATIONSHIP_TYPE",
      "target": "Entity Name B",
      "description": "One sentence explaining why this relationship exists."
    }}
  ]
}}"""


REFLECTION_PROMPT = """You are reviewing a knowledge graph extraction from a TIET policy document.

ORIGINAL TEXT:
\"\"\"
{chunk_text}
\"\"\"

EXTRACTED ENTITIES AND RELATIONSHIPS:
{extracted}

TASK: Check if anything important was MISSED. Look specifically for:
1. Any committees, roles, or authorities mentioned but not extracted
2. Any conditions, deadlines, or exceptions that were skipped
3. Any relationships between extracted entities that were missed
4. Any consequences or requirements not captured

If extraction is complete, respond: {{"complete": true, "missing": []}}
If something is missing, respond:
{{
  "complete": false,
  "missing_entities": [
    {{"name": "...", "type": "...", "description": "..."}}
  ],
  "missing_relationships": [
    {{"source": "...", "relation": "...", "target": "...", "description": "..."}}
  ]
}}

Output ONLY valid JSON. Do not include markdown. Wrong keys or extra text will be rejected."""


def extract_entities_and_relations(chunk):
    """
    Two-pass extraction: extract → reflect → merge → validate.
    Returns dict with entities and relationships lists.
    Retries once if validation yields too few valid items.
    """
    prompt = ENTITY_EXTRACTION_PROMPT.format(
        doc_name=chunk['document_name'],
        category=chunk['category'],
        section=chunk.get('section_title', 'General'),
        chunk_text=chunk['chunk_text'][:2000]
    )

    # Pass 1: Extract
    raw = llm_call(prompt, max_tokens=1500, temp=0.1)
    result = safe_json_parse(raw)

    if not result or not isinstance(result, dict) or 'entities' not in result:
        log.warning("  [extract] No valid JSON for %s — raw output:\n%s", chunk['chunk_id'], raw[:1000])
        raw = llm_call(prompt, max_tokens=1500, temp=0.1)
        result = safe_json_parse(raw)
        if not result or not isinstance(result, dict) or 'entities' not in result:
            log.warning("  [extract] Retry also failed for %s — raw output:\n%s", chunk['chunk_id'], raw[:1000])
            return {"entities": [], "relationships": []}

    entities = result.get('entities', [])
    relations = result.get('relationships', [])

    # Pass 2: Reflection (discriminator)
    if len(entities) > 0:
        reflection_prompt = REFLECTION_PROMPT.format(
            chunk_text=chunk['chunk_text'][:1500],
            extracted=json.dumps({"entities": entities[:10], "relationships": relations[:8]}, indent=2)
        )
        refl_raw = llm_call(reflection_prompt, max_tokens=800, temp=0.1)
        refl = safe_json_parse(refl_raw)

        if refl and isinstance(refl, dict) and not refl.get('complete', True):
            entities += refl.get('missing_entities', [])
            relations += refl.get('missing_relationships', [])

    # Strict validation (Fix 2)
    validated = validate_and_filter_extraction({"entities": entities, "relationships": relations})

    # If validation killed everything and we had raw entities, retry once
    if len(validated['entities']) == 0 and len(entities) > 0:
        log.info("  [extract] All entities invalidated for %s — retrying extraction", chunk['chunk_id'])
        raw = llm_call(prompt, max_tokens=1500, temp=0.1)
        result = safe_json_parse(raw)
        if result and isinstance(result, dict) and 'entities' in result:
            validated = validate_and_filter_extraction(result)

    return validated

In [23]:
# ================================================================
# CELL D — BUILD KNOWLEDGE GRAPH (NetworkX) (Fix 3)
# Nodes = entities, Edges = relationships
# Advanced provenance, deduplication, and alias tracking.
# ================================================================

import networkx as nx
from sentence_transformers import SentenceTransformer
import numpy as np
import string

# Load embedder (CPU)
log.info("Loading %s embedder...", EMBEDDER_MODEL)
embedder = SentenceTransformer(EMBEDDER_MODEL, device='cuda')
log.info("Embedder loaded")

# Build graph
G = nx.MultiDiGraph()
name_index = {}   # canonical_name -> node_id
alias_index = {}  # alias -> node_id

# Store raw extraction results for debugging
extraction_log = []

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
GRAPH_CKPT = f"{CHECKPOINT_DIR}/graph_checkpoint.json"

STOPWORDS = {"the", "of", "and", "a", "an", "for", "in", "to", "on", "at"}

def normalize_entity_name(name: str) -> str:
    """Advanced normalization: lowercase, strip punctuation, remove trivial stopwords."""
    name = name.lower().strip()
    # Remove punctuation
    name = name.translate(str.maketrans('', '', string.punctuation))
    # Collapse whitespace
    words = name.split()
    # Remove stopwords only if it leaves something meaningful
    filtered = [w for w in words if w not in STOPWORDS]
    if len(filtered) == 0 and len(words) > 0:
        filtered = words  # Fallback if name was just "the and"
    return " ".join(filtered)

# Alias for backwards compatibility with Cells 21, 22
normalize_name = normalize_entity_name

def jaccard_similarity(s1: str, s2: str) -> float:
    set1, set2 = set(s1.split()), set(s2.split())
    if not set1 or not set2: return 0.0
    return len(set1 & set2) / len(set1 | set2)

def resolve_canonical_entity(name: str, etype: str) -> str:
    """
    Layered resolution:
    1. Exact normalized match
    2. Alias match
    3. Lexical approximate match (Jaccard > 0.7) for same entity type.
    Returns existing node_id or None.
    """
    norm = normalize_entity_name(name)
    if not norm: return None

    # 1. Exact match in primary index
    if norm in name_index:
        return name_index[norm]

    # 2. Alias match
    if norm in alias_index:
        return alias_index[norm]

    # 3. Approximate lexical match (only if same type)
    best_match = None
    best_score = 0.0
    for cand_norm, cand_id in name_index.items():
        if cand_id not in G: continue
        cand_type = G.nodes[cand_id].get("type")
        if cand_type != etype:
            continue
        score = jaccard_similarity(norm, cand_norm)
        if score > 0.75 and score > best_score:
            best_score = score
            best_match = cand_id

    return best_match

def add_entity_to_graph(ent: dict, chunk: dict):
    """Add or update a node with rich provenance."""
    raw_name = ent.get('name')
    etype = ent.get('type')
    if not raw_name or not etype: return None

    norm = normalize_entity_name(raw_name)
    if not norm or len(norm) < ENTITY_MIN_NAME_LEN: return None

    node_id = resolve_canonical_entity(raw_name, etype)
    chunk_id = chunk['chunk_id']
    doc_name = chunk['document_name']
    section = chunk.get('section_title', 'General')
    desc = ent.get('description', '')
    conf = ent.get('confidence', 1.0)

    if node_id is None:
        # Create new node
        node_id = f"{etype}::{norm[:60]}"
        G.add_node(node_id,
            name=raw_name,
            canonical_name=norm,
            aliases=[norm, raw_name],
            type=etype,
            description=desc,
            source_chunks=[chunk_id],
            source_docs=[doc_name],
            sections=[section],
            first_seen_chunk=chunk_id,
            last_seen_chunk=chunk_id,
            confidence=conf
        )
        name_index[norm] = node_id
        alias_index[norm] = node_id
        alias_index[raw_name.lower().strip()] = node_id
    else:
        # Update existing node
        d = G.nodes[node_id]
        if norm not in d.get('aliases', []):
            d.setdefault('aliases', []).append(norm)
            alias_index[norm] = node_id
        if raw_name not in d.get('aliases', []):
            d['aliases'].append(raw_name)
            alias_index[raw_name.lower().strip()] = node_id

        if chunk_id not in d.get('source_chunks', []):
            d.setdefault('source_chunks', []).append(chunk_id)
        if doc_name not in d.get('source_docs', []):
            d.setdefault('source_docs', []).append(doc_name)
        if section not in d.get('sections', []):
            d.setdefault('sections', []).append(section)

        d['last_seen_chunk'] = chunk_id

        # Update description if it adds info
        if desc and desc not in d.get('description', ''):
            d['description'] = d.get('description', '') + ' | ' + desc

        # Update confidence
        if conf > d.get('confidence', 0.0):
            d['confidence'] = conf

    return node_id


def add_relationship_to_graph(rel: dict, chunk: dict, is_ld: bool = False):
    """Add or update an edge with rich provenance."""
    src_raw = rel.get('source')
    tgt_raw = rel.get('target')
    rel_type = rel.get('relation', 'RELATED_TO')

    # Resolve using alias/canonical map (type is unknown here, so we pass None to skip strict type checking during resolution, or rely entirely on exact/alias matches)
    # Actually, we can just look them up directly since they should have been added as entities just before this.
    src_norm = normalize_entity_name(src_raw)
    tgt_norm = normalize_entity_name(tgt_raw)

    src_id = name_index.get(src_norm) or alias_index.get(src_norm)
    tgt_id = name_index.get(tgt_norm) or alias_index.get(tgt_norm)

    # Fallback to exact alias if still none
    if not src_id: src_id = alias_index.get(src_raw.lower().strip())
    if not tgt_id: tgt_id = alias_index.get(tgt_raw.lower().strip())

    if src_id and tgt_id and src_id != tgt_id:
        if not G.has_edge(src_id, tgt_id, key=rel_type):
            G.add_edge(src_id, tgt_id, key=rel_type,
                relation=rel_type,
                description=rel.get('description', ''),
                source_chunk=chunk['chunk_id'],
                source_doc=chunk['document_name'],
                source_section=chunk.get('section_title', 'General'),
                is_long_distance=is_ld,
                confidence=rel.get('confidence', 1.0)
            )
            return True
    return False

def add_to_graph(chunk: dict, extraction: dict) -> None:
    """Wrapper to add all entities and relationships."""
    for ent in extraction.get('entities', []):
        add_entity_to_graph(ent, chunk)
    for rel in extraction.get('relationships', []):
        add_relationship_to_graph(rel, chunk, is_ld=False)


# Load checkpoint if exists
done_chunks = set()
if os.path.exists(GRAPH_CKPT):
    with open(GRAPH_CKPT) as f:
        ckpt = json.load(f)
    # Restore graph
    for node_id, node_data in ckpt.get('nodes', {}).items():
        G.add_node(node_id, **node_data)
    for edge in ckpt.get('edges', []):
        edge_data = dict(edge)
        key = edge_data.pop('key', edge_data.get('relation'))
        G.add_edge(edge_data['source'], edge_data['target'], key=key, **edge_data)

    done_chunks = set(ckpt.get('done_chunks', []))

    # Rebuild indices
    for n, d in G.nodes(data=True):
        if 'canonical_name' in d:
            name_index[d['canonical_name']] = n
        for al in d.get('aliases', []):
            alias_index[al] = n
            alias_index[al.lower().strip()] = n

    log.info("Resumed: %d nodes, %d edges, %d chunks done", G.number_of_nodes(), G.number_of_edges(), len(done_chunks))


def save_graph_checkpoint() -> None:
    """Save graph state to JSON checkpoint."""
    ckpt = {
        "nodes": {n: dict(d) for n, d in G.nodes(data=True)},
        "edges": [
            {
                "source": u, "target": v, "key": k,
                **d
            }
            for u, v, k, d in G.edges(keys=True, data=True)
        ],
        "done_chunks": list(done_chunks)
    }
    with open(GRAPH_CKPT, 'w') as f:
        json.dump(ckpt, f, ensure_ascii=False)


log.info("Building Knowledge Graph")
log.info("Total chunks to process: %d", len(chunk_registry))
log.info("Already done: %d", len(done_chunks))

error_count = 0
success_count = 0
for i, chunk in enumerate(chunk_registry):
    if chunk['chunk_id'] in done_chunks:
        continue
    if len(chunk['chunk_text']) < MIN_CHUNK_LENGTH:
        done_chunks.add(chunk['chunk_id'])
        continue

    try:
        extraction = extract_entities_and_relations(chunk)
        add_to_graph(chunk, extraction)
        extraction_log.append({
            "chunk_id": chunk['chunk_id'],
            "doc": chunk['document_name'],
            "n_entities": len(extraction.get('entities', [])),
            "n_relations": len(extraction.get('relationships', []))
        })
        done_chunks.add(chunk['chunk_id'])
        success_count += 1

        if i % 10 == 0:
            log.info("  [%d/%d] Nodes: %d | Edges: %d", i, len(chunk_registry), G.number_of_nodes(), G.number_of_edges())
            save_graph_checkpoint()

    except Exception as e:
        error_count += 1
        log.warning("Error on %s: %s", chunk['chunk_id'], e)
        done_chunks.add(chunk['chunk_id'])
        continue

# Final save
save_graph_checkpoint()
log.info("Extraction done: %d/%d chunks succeeded, %d errors", success_count, len(chunk_registry), error_count)
log.info("Graph built: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
{
  "entities": [
    {
      "name": "GATE Score",
      "type": "ELIGIBILITY",
      "description": "The score required for admission to M.E./M.Tech. programs based on the Graduate Aptitude Test in Engineering.",
      "source_relation": ""
    },
    {
      "name": "60% Aggregate Marks",
      "type": "CONDITION",
      "description": "The minimum aggregate marks required in the qualifying examination from a recognized university for admission.",
      "source_relation": ""
    },
    {
      "name": "12th Marks",
      type: "CONDITION",
      description: "The marks from the 12th standard that contribute to the merit calculation for admission.",
      source_relation: ""
    },
    {
      "name": "B.E./B.Tech. Marks",
      "type":

In [25]:
# ================================================================
# CELL E — LONG-DISTANCE RELATIONSHIP EXTRACTION (Semantic IoU)
# Find related chunks across different documents using embeddings.
# This is the HSG-RAG innovation that catches cross-policy links.
# ================================================================

log.info("Computing chunk embeddings for cross-chunk linking")
EMBED_CACHE = f"{CHECKPOINT_DIR}/chunk_embeddings.npy"
EMBED_IDS   = f"{CHECKPOINT_DIR}/chunk_ids.json"  # (defined in CONFIG)

if os.path.exists(EMBED_CACHE):
    chunk_embeddings = np.load(EMBED_CACHE)
    with open(EMBED_IDS) as f:
        embed_chunk_ids = json.load(f)
    log.info("Loaded cached embeddings: %s", chunk_embeddings.shape)
else:
    texts = [
        f"{c.get('section_title','General')} {c['chunk_text'][:400]}"
        for c in chunk_registry
    ]
    chunk_embeddings = embedder.encode(
        texts, batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    embed_chunk_ids = [c['chunk_id'] for c in chunk_registry]
    np.save(EMBED_CACHE, chunk_embeddings)
    with open(EMBED_IDS, 'w') as f:
        json.dump(embed_chunk_ids, f)
    log.info("Embeddings computed: %s", chunk_embeddings.shape)


LONG_DIST_PROMPT = """You are analyzing TWO policy text chunks from TIET documents to find relationships between them.

CHUNK 1 [{doc1} | {section1}]:
\"\"\"
{text1}
\"\"\"

CHUNK 2 [{doc2} | {section2}]:
\"\"\"
{text2}
\"\"\"

TASK: Identify meaningful CROSS-CHUNK relationships between entities in these two chunks.
Only extract relationships that genuinely link entities across both chunks.
Skip if the chunks are unrelated.

Output ONLY valid JSON. Do not include markdown. Wrong keys or extra text will be rejected.
{{
  "has_relationships": true,
  "relationships": [
    {{
      "source": "Entity from chunk 1",
      "relation": "RELATIONSHIP_TYPE",
      "target": "Entity from chunk 2",
      "description": "Why these are related across documents."
    }}
  ]
}}"""


def cosine_similarity(emb_a: np.ndarray, emb_b: np.ndarray) -> float:
    """Cosine similarity between two embedding vectors."""
    return float(np.dot(emb_a, emb_b))


def find_and_extract_long_distance_relations(chunk_registry: list, embeddings: np.ndarray, max_pairs: int = 200) -> None:
    """
    Find cross-chunk pairs using Semantic IoU, extract relationships.
    """
    log.info("Finding cross-chunk pairs (max %d)...", max_pairs)
    pairs_found = []
    n = len(chunk_registry)

    # Sample every 3rd chunk as anchor
    for i in range(0, n, 3):
        sims = embeddings[i] @ embeddings.T
        # Find related-but-not-identical chunks (0.45 < sim < 0.85)
        candidates = [
            j for j in np.where((sims > SIM_LOW) & (sims < SIM_HIGH))[0]
            if j != i and chunk_registry[j]['document_name'] != chunk_registry[i]['document_name']
        ]
        if candidates:
            # Take top 2 most similar cross-doc chunks
            candidates = sorted(candidates, key=lambda j: sims[j], reverse=True)[:LONG_DISTANCE_TOP_K]
            for j in candidates:
                pairs_found.append((i, j))

        if len(pairs_found) >= max_pairs:
            break

    log.info("  Found %d cross-document pairs to analyze", len(pairs_found))

    ld_relations_added = 0
    for idx, (i, j) in enumerate(pairs_found[:max_pairs]):
        c1, c2 = chunk_registry[i], chunk_registry[j]
        try:
            prompt = LONG_DIST_PROMPT.format(
                doc1=c1['document_name'], section1=c1.get('section_title','General'),
                text1=c1['chunk_text'][:LD_PROMPT_MAX_TEXT],
                doc2=c2['document_name'], section2=c2.get('section_title','General'),
                text2=c2['chunk_text'][:LD_PROMPT_MAX_TEXT],
            )
            raw = llm_call_with_retry(prompt, max_tokens=LLM_MAX_TOKENS_SHORT, temp=0.1)
            result = safe_json_parse(raw)

            if result and result.get('has_relationships') and result.get('relationships'):
                for rel in result['relationships']:
                    # Fix 2: Validate each relationship before adding
                    if not validate_relationship(rel):
                        log.info("  [LD] Dropped invalid relation: %s", rel.get('relation', '?'))
                        continue

                    # Fix 3: Add relationship using rich provenance
                    # Passing c1 as the chunk context for this edge
                    added = add_relationship_to_graph(rel, c1, is_ld=True)
                    if added:
                        ld_relations_added += 1

            if idx % 20 == 0:
                log.info("  [%d/%d] Long-distance relations added: %d", idx, len(pairs_found), ld_relations_added)
                save_graph_checkpoint()

        except Exception as e:
            log.warning("Long-distance extraction error: %s", e)
            continue

    log.info("Long-distance extraction done. Added %d cross-document edges", ld_relations_added)
    save_graph_checkpoint()


find_and_extract_long_distance_relations(chunk_registry, chunk_embeddings, max_pairs=MAX_LONG_DISTANCE_PAIRS)

print(f"\n✓ FINAL GRAPH: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Final checkpoint and graph export
save_graph_checkpoint()

# Save final graph as .gpickle
import pickle
graph_path = f"{CHECKPOINT_DIR}/tiet_knowledge_graph.gpickle"
with open(graph_path, 'wb') as f:
    pickle.dump(G, f)
log.info("Graph saved: %s", graph_path)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


✓ FINAL GRAPH: 53 nodes, 24 edges


In [ ]:
# ================================================================
# CELL F — QA DATASET GENERATION
# 5 question types: single-hop, cross-chunk, 2-hop, 3-hop, 4-hop
# Non-generic prompts that READ the actual text content
# ================================================================

# QA_DATASET_DIR is defined in CONFIG cell
os.makedirs(QA_DATASET_DIR, exist_ok=True)
QA_CHECKPOINT = f"{QA_DATASET_DIR}/qa_checkpoint.json"

# Load QA checkpoint
qa_dataset = []
if os.path.exists(QA_CHECKPOINT):
    with open(QA_CHECKPOINT) as f:
        qa_dataset = json.load(f)
    log.info("Resumed QA dataset: %d pairs", len(qa_dataset))

done_qa_chunks = set()
for qa in qa_dataset:
    for cid in qa.get('source_chunks', []):
        done_qa_chunks.add(cid)

def save_qa_checkpoint():
    with open(QA_CHECKPOINT, 'w', encoding='utf-8') as f:
        json.dump(qa_dataset, f, indent=2, ensure_ascii=False)


# ---- SINGLE-HOP PROMPT ----------------------------------------
SINGLE_HOP_PROMPT = """You are creating a high-quality QA dataset from TIET (Thapar Institute of Engineering & Technology) policy documents.

SOURCE: {doc_name} | Section: {section} | Category: {category}

POLICY TEXT:
\"\"\"
{chunk_text}
\"\"\"

Generate exactly 3 single-hop questions. Rules:
- Each question must be strictly answerable from THIS TEXT ALONE.
- Questions must be SPECIFIC — use actual names, numbers, percentages, dates from the text.
- Do NOT ask vague questions like "What is this document about?" or "What are the rules?".
- Answers must be precise (2-4 sentences), quoting specific details directly.
- Include the exact snippet of "evidence" from the text that proves the answer.
- Use a DIFFERENT question type for each:
  * Type 1: rule_lookup (what is the specific rule/requirement?)
  * Type 2: eligibility_criteria (who qualifies / what conditions apply?)
  * Type 3: procedure_step (what is the process/how to do X?)

Output ONLY valid JSON array. Do not include markdown. Use exactly the fields specified.
Wrong keys, missing fields, or extra text will be rejected. Return an empty list if nothing valid exists.
[
  {{
    "question": "Specific question using actual terms from the text?",
    "answer": "Precise answer with specific details and numbers from the text.",
    "question_type": "rule_lookup",
    "difficulty": "easy",
    "key_facts": ["fact1", "fact2"],
    "evidence": "Exact quoted sentence from the text supporting the answer."
  }},
  {{
    "question": "...",
    "answer": "...",
    "question_type": "eligibility_criteria",
    "difficulty": "medium",
    "key_facts": [],
    "evidence": "..."
  }},
  {{
    "question": "...",
    "answer": "...",
    "question_type": "procedure_step",
    "difficulty": "medium",
    "key_facts": [],
    "evidence": "..."
  }}
]"""


# ---- CROSS-CHUNK PROMPT (same doc, adjacent sections) ----------
CROSS_CHUNK_PROMPT = """You are creating multi-hop QA pairs from TIET policy documents.

These sections are from the SAME document: {doc_name}

SECTION A [{section_a}]:
\"\"\"
{text_a}
\"\"\"

SECTION B [{section_b}]:
\"\"\"
{text_b}
\"\"\"

Generate 2 cross-section questions where:
- The COMPLETE answer requires BOTH sections.
- Neither section alone can answer the question. If it can be answered by one section, DO NOT output it.
- The question clearly links concepts from Section A and Section B.
- Do not ask vague questions.

Focus on: how a rule in A is qualified/implemented in B, or conditions in A whose consequences are in B.

Output ONLY valid JSON array. Do not include markdown. Use exactly the fields specified.
Wrong keys, missing fields, or extra text will be rejected. Return an empty list if nothing valid exists.
[
  {{
    "question": "Question requiring both sections?",
    "answer": "Complete answer explicitly combining info from both sections.",
    "reasoning_chain": [
      "From Section A ({section_a}): specific fact...",
      "From Section B ({section_b}): specific fact...",
      "Combined: conclusion..."
    ],
    "question_type": "cross_chunk",
    "difficulty": "medium",
    "hop_count": 2
  }}
]"""


# ---- MULTI-HOP PROMPT (2/3/4 hop, cross-document) -------------
MULTI_HOP_PROMPT = """You are creating {hop_count}-hop QA pairs from TIET institutional documents.

A {hop_count}-hop question CANNOT be answered from any single passage — it requires synthesizing information from ALL {hop_count} passages below.

{passages}

Generate 2 multi-hop questions that:
1. CANNOT be answered from any single passage alone. If one passage is enough, discard the question.
2. Require reasoning across ALL {hop_count} passages.
3. Test real understanding of how TIET policies/rules interact.
4. Have explicit, logical reasoning chains showing which info comes from where.

Difficulty for {hop_count}-hop: {difficulty}

Output ONLY valid JSON array. Do not include markdown. Use exactly the fields specified.
Wrong keys, missing fields, or extra text will be rejected. Return an empty list if nothing valid exists.
[
  {{
    "question": "Question requiring all {hop_count} passages?",
    "answer": "Complete synthesized answer with specific details from each passage.",
    "reasoning_chain": [
      "Step 1 - From [source]: specific fact...",
      "Step 2 - From [source]: specific fact...",
      "Step 3 - Combining: conclusion..."
    ],
    "question_type": "cross_document_{hop_count}hop",
    "difficulty": "{difficulty}",
    "hop_count": {hop_count},
    "is_cross_document": true,
    "sources_needed": {sources_list}
  }}
]"""


def format_passages(chunk_group):
    """Format chunk group into numbered passages for prompt."""
    out = ""
    for i, c in enumerate(chunk_group, 1):
        out += f"\nPASSAGE {i} [{c['document_name']} | {c.get('section_title','General')}]:\n"
        out += f"\"\"\"\n{c['chunk_text'][:650]}\n\"\"\"\n"
    return out


def generate_single_hop(chunk):
    if len(chunk['chunk_text']) < 120:
        return []
    prompt = SINGLE_HOP_PROMPT.format(
        doc_name=chunk['document_name'],
        section=chunk.get('section_title','General'),
        category=chunk['category'],
        chunk_text=chunk['chunk_text'][:1800]
    )
    raw = llm_call(prompt, max_tokens=900, temp=0.2)
    result = safe_json_parse(raw)
    if not result or not isinstance(result, list):
        return []
    # Fix 2: Validate QA items before adding metadata
    result = validate_and_filter_qa_list(result, expected_hop_type="single_hop")
    if not result:
        return []
    for qa in result:
        qa.update({
            "hop_type": "single_hop",
            "hop_count": 1,
            "source_chunks": [chunk['chunk_id']],
            "categories": [chunk['category']],
            "sources": [chunk['document_name']],
            "source_sections": [chunk.get('section_title','General')],
            "is_cross_document": False,
            "is_cross_chunk": False,
        })
    return result


def generate_cross_chunk(c1, c2):
    """Adjacent chunks from same document."""
    prompt = CROSS_CHUNK_PROMPT.format(
        doc_name=c1['document_name'],
        section_a=c1.get('section_title','General'),
        text_a=c1['chunk_text'][:750],
        section_b=c2.get('section_title','General'),
        text_b=c2['chunk_text'][:750],
    )
    raw = llm_call(prompt, max_tokens=900, temp=0.2)
    result = safe_json_parse(raw)
    if not result or not isinstance(result, list):
        return []
    # Fix 2: Validate QA items before adding metadata
    result = validate_and_filter_qa_list(result, expected_hop_type="cross_chunk")
    if not result:
        return []
    for qa in result:
        qa.update({
            "hop_type": "cross_chunk",
            "hop_count": 2,
            "source_chunks": [c1['chunk_id'], c2['chunk_id']],
            "categories": list(set([c1['category'], c2['category']])),
            "sources": [c1['document_name']],
            "source_sections": [c1.get('section_title','General'), c2.get('section_title','General')],
            "is_cross_document": False,
            "is_cross_chunk": True,
        })
    return result


def generate_multi_hop(chunk_group, hop_count):
    """Cross-document multi-hop questions."""
    passages = format_passages(chunk_group)
    sources_list = json.dumps(list(set(c['document_name'] for c in chunk_group)))
    difficulty = "medium" if hop_count == 2 else "hard"

    prompt = MULTI_HOP_PROMPT.format(
        hop_count=hop_count,
        passages=passages,
        sources_list=sources_list,
        difficulty=difficulty,
    )
    raw = llm_call(prompt, max_tokens=1100, temp=0.25)
    result = safe_json_parse(raw)
    if not result or not isinstance(result, list):
        return []
    # Fix 2: Validate QA items before adding metadata
    result = validate_and_filter_qa_list(result, expected_hop_type=f"{hop_count}_hop")
    if not result:
        return []
    for qa in result:
        qa.update({
            "hop_type": f"{hop_count}_hop",
            "hop_count": hop_count,
            "source_chunks": [c['chunk_id'] for c in chunk_group],
            "categories": list(set(c['category'] for c in chunk_group)),
            "sources": list(set(c['document_name'] for c in chunk_group)),
            "source_sections": [c.get('section_title','General') for c in chunk_group],
            "is_cross_document": True,
            "is_cross_chunk": False,
        })
    return result


# ================================================================
# CELL G — RUN QA GENERATION
# ================================================================

import pandas as pd
from datetime import datetime

def find_multi_hop_groups(chunk_registry, embeddings, n_hops, max_groups=120):
    """Find chunk groups for multi-hop QA using semantic similarity."""
    groups = []
    n = len(chunk_registry)

    for i in range(0, n, 4):
        if len(groups) >= max_groups:
            break
        sims = embeddings[i] @ embeddings.T
        # Cross-doc candidates: related but not same doc
        cands = [
            j for j in np.argsort(sims)[::-1]
            if j != i
            and chunk_registry[j]['document_name'] != chunk_registry[i]['document_name']
            and 0.35 < sims[j] < 0.82
        ]
        if len(cands) < n_hops - 1:
            continue
        group = [chunk_registry[i]] + [chunk_registry[j] for j in cands[:n_hops-1]]
        groups.append(group)

    return groups


log.info("\n═══════════════════════════════════════")
log.info("  PHASE 1: Single-hop QA Generation")
log.info("═══════════════════════════════════════")

sampled_chunks = [c for i, c in enumerate(chunk_registry) if i % 2 == 0]  # every 2nd chunk

for idx, chunk in enumerate(tqdm(sampled_chunks, desc="Single-hop")):
    if chunk['chunk_id'] in done_qa_chunks:
        continue
    try:
        qas = generate_single_hop(chunk)
        qa_dataset.extend(qas)
        done_qa_chunks.add(chunk['chunk_id'])
        if idx % 15 == 0:
            save_qa_checkpoint()
            log.info("  Checkpoint: %s QA pairs saved", len(qa_dataset))
    except Exception as e:
        log.warning("Error: %s", e)

save_qa_checkpoint()
log.info("Single-hop done: %s pairs", sum(1 for q in qa_dataset if q.get('hop_type')=='single_hop'))

gc.collect()

log.info("\n═══════════════════════════════════════")
log.info("  PHASE 2: Cross-chunk QA Generation")
log.info("═══════════════════════════════════════")

# Group by document, find adjacent chunk pairs
doc_chunks = {}
for c in chunk_registry:
    doc_chunks.setdefault(c['document_name'], []).append(c)

cross_chunk_pairs = []
for doc, chunks in doc_chunks.items():
    if len(chunks) < 2:
        continue
    # Pair consecutive chunks within each document
    for k in range(len(chunks) - 1):
        cross_chunk_pairs.append((chunks[k], chunks[k+1]))
    # Also pair every 3rd chunk for variety
    for k in range(0, len(chunks)-2, 3):
        cross_chunk_pairs.append((chunks[k], chunks[k+2]))

log.info("Cross-chunk pairs found: %s", len(cross_chunk_pairs))

for idx, (c1, c2) in enumerate(tqdm(cross_chunk_pairs[:120], desc="Cross-chunk")):
    pair_key = f"{c1['chunk_id']}+{c2['chunk_id']}"
    if pair_key in done_qa_chunks:
        continue
    try:
        qas = generate_cross_chunk(c1, c2)
        qa_dataset.extend(qas)
        done_qa_chunks.add(pair_key)
        if idx % 15 == 0:
            save_qa_checkpoint()
    except Exception as e:
        log.warning("Error: %s", e)

save_qa_checkpoint()
log.info("Cross-chunk done: %s pairs", sum(1 for q in qa_dataset if q.get('hop_type')=='cross_chunk'))

gc.collect()

log.info("\n═══════════════════════════════════════")
log.info("  PHASE 3: 2-hop Cross-document QA")
log.info("═══════════════════════════════════════")

groups_2hop = find_multi_hop_groups(chunk_registry, chunk_embeddings, n_hops=2, max_groups=100)
for idx, group in enumerate(tqdm(groups_2hop, desc="2-hop")):
    gkey = "+".join(c['chunk_id'] for c in group)
    if gkey in done_qa_chunks:
        continue
    try:
        qas = generate_multi_hop(group, hop_count=2)
        qa_dataset.extend(qas)
        done_qa_chunks.add(gkey)
        if idx % 15 == 0:
            save_qa_checkpoint()
    except Exception as e:
        log.warning("Error: %s", e)

save_qa_checkpoint()
log.info("2-hop done: %s pairs", sum(1 for q in qa_dataset if q.get('hop_type')=='2_hop'))

gc.collect()

log.info("\n═══════════════════════════════════════")
log.info("  PHASE 4: 3-hop Cross-document QA")
log.info("═══════════════════════════════════════")

groups_3hop = find_multi_hop_groups(chunk_registry, chunk_embeddings, n_hops=3, max_groups=60)
for idx, group in enumerate(tqdm(groups_3hop, desc="3-hop")):
    gkey = "+".join(c['chunk_id'] for c in group)
    if gkey in done_qa_chunks:
        continue
    try:
        qas = generate_multi_hop(group, hop_count=3)
        qa_dataset.extend(qas)
        done_qa_chunks.add(gkey)
        if idx % 10 == 0:
            save_qa_checkpoint()
    except Exception as e:
        log.warning("Error: %s", e)

save_qa_checkpoint()
log.info("3-hop done: %s pairs", sum(1 for q in qa_dataset if q.get('hop_type')=='3_hop'))

gc.collect()

log.info("\n═══════════════════════════════════════")
log.info("  PHASE 5: 4-hop Cross-document QA")
log.info("═══════════════════════════════════════")

groups_4hop = find_multi_hop_groups(chunk_registry, chunk_embeddings, n_hops=4, max_groups=30)
for idx, group in enumerate(tqdm(groups_4hop, desc="4-hop")):
    gkey = "+".join(c['chunk_id'] for c in group)
    if gkey in done_qa_chunks:
        continue
    try:
        qas = generate_multi_hop(group, hop_count=4)
        qa_dataset.extend(qas)
        done_qa_chunks.add(gkey)
        if idx % 10 == 0:
            save_qa_checkpoint()
    except Exception as e:
        log.warning("Error: %s", e)

save_qa_checkpoint()


In [27]:
#================================================================
# CELL H — SAVE FINAL DATASET (JSON + CSV) + GROUNDEDNESS + GRAPH EXPORT
# ================================================================

def verify_answer_support(qa_pair: dict, chunk_registry: list, embedder, threshold: float = 0.65) -> dict:
    """Fix 4: Verify answer is grounded using lexical overlap and embedding similarity."""
    answer = qa_pair.get("answer", "")
    source_chunk_ids = qa_pair.get("source_chunks", [])

    if not answer or not source_chunk_ids:
        qa_pair["grounded"] = False
        qa_pair["grounded_score"] = 0.0
        qa_pair["matched_evidence"] = ""
        return qa_pair

    matches = [c for c in chunk_registry if c["chunk_id"] in source_chunk_ids]
    if not matches:
        qa_pair["grounded"] = False
        qa_pair["grounded_score"] = 0.0
        qa_pair["matched_evidence"] = ""
        return qa_pair

    # Combine all relevant text
    combined_text = " ".join([c["chunk_text"] for c in matches])

    # Use explicit evidence if the model provided it
    evidence = qa_pair.get("evidence", "")
    if evidence and len(evidence) > 10:
        # Check if the generated evidence actually exists in the source
        # using simple substring or Jaccard
        combined_lower = combined_text.lower()
        ev_lower = evidence.lower()
        if ev_lower in combined_lower:
            qa_pair["grounded"] = True
            qa_pair["grounded_score"] = 1.0
            qa_pair["matched_evidence"] = evidence
            return qa_pair

    # 1. Lexical overlap check (Jaccard on words)
    def jaccard(s1, s2):
        set1 = set(re.findall(r'\w+', s1.lower()))
        set2 = set(re.findall(r'\w+', s2.lower()))
        if not set1 or not set2: return 0.0
        return len(set1 & set2) / len(set1 | set2)

    # Splitting combined text into rough sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?]) +', combined_text) if len(s.strip()) > 10]

    best_sent = ""
    best_lex = 0.0
    for sent in sentences:
        score = jaccard(answer, sent)
        if score > best_lex:
            best_lex = score
            best_sent = sent

    # Defer embedding to batch step
    qa_pair["_best_lex"] = best_lex
    qa_pair["_best_sent"] = best_sent
    qa_pair["_combined_text"] = combined_text
    return qa_pair

log.info("Running advanced lexical groundedness check on %d QA pairs...", len(qa_dataset))
qa_dataset = [verify_answer_support(q, chunk_registry, embedder, threshold=GROUNDEDNESS_SIM_THRESHOLD) for q in qa_dataset]

# Batch process embeddings for semantic fallback
log.info("Running batched semantic fallback for grounding...")
answers_to_encode = []
contexts_to_encode = []
indices_to_encode = []

for i, qa in enumerate(qa_dataset):
    lex_score = qa.pop("_best_lex", 0.0)
    best_sent = qa.pop("_best_sent", "")
    context = qa.pop("_combined_text", "")

    if lex_score > 0.25:
        # Lexical overlap is strong enough
        qa["grounded"] = True
        qa["grounded_score"] = 1.0
        qa["matched_evidence"] = best_sent
    elif qa.get("grounded", False):
        # Already grounded via explicit generated evidence string in step 1
        pass
    else:
        # Need semantic fallback
        answers_to_encode.append(qa.get("answer", ""))
        contexts_to_encode.append(context)
        indices_to_encode.append(i)

        qa["grounded"] = False
        qa["grounded_score"] = 0.0
        qa["matched_evidence"] = ""

if answers_to_encode:
    emb_answers = embedder.encode(answers_to_encode, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
    emb_contexts = embedder.encode(contexts_to_encode, batch_size=32, normalize_embeddings=True, show_progress_bar=False)

    # Calculate cosine similarity per pair
    import numpy as np
    sims = np.sum(emb_answers * emb_contexts, axis=1)

    for idx, sim in zip(indices_to_encode, sims):
        sim = float(sim)
        qa = qa_dataset[idx]
        if sim >= GROUNDEDNESS_SIM_THRESHOLD:
            qa["grounded"] = True
            qa["grounded_score"] = round(sim, 4)

grounded_count = sum(1 for q in qa_dataset if q.get("grounded"))
log.info("Groundedness: %d/%d pairs pass (threshold=%.2f)", grounded_count, len(qa_dataset), GROUNDEDNESS_SIM_THRESHOLD)


# ── Graph export ──
# GEXF: collapse MultiDiGraph -> DiGraph (multi-edges lose key)
graph_gexf = f"{CHECKPOINT_DIR}/tiet_knowledge_graph.gexf"
try:
    G_simple = nx.DiGraph()
    for u, v, d in G.edges(data=True):
        if not G_simple.has_edge(u, v):
            G_simple.add_edge(u, v, **d)
    nx.write_gexf(G_simple, graph_gexf)
    log.info("Graph exported: %s", graph_gexf)
except Exception as e:
    log.warning("GEXF export failed: %s", e)

# GraphML: convert list attributes to pipe-joined strings first
graph_graphml = f"{CHECKPOINT_DIR}/tiet_knowledge_graph.graphml"
try:
    G_export = G.copy()
    for _, d in G_export.nodes(data=True):
        for k in list(d):
            if isinstance(d[k], list):
                d[k] = " | ".join(str(x) for x in d[k])
    nx.write_graphml(G_export, graph_graphml)
    log.info("Graph exported: %s", graph_graphml)
except Exception as e:
    log.warning("GraphML export failed: %s", e)


# ── Dedup (removes semantically duplicate questions) ──
def dedup_questions(qa_dataset: list, embedder, threshold: float = 0.92) -> list:
    """Fix 4: Advanced deduplication based on semantics + hop_type + exact string."""
    keep = []
    kept_embs = []
    kept_hop_types = []
    kept_exact = set()

    questions = [qa.get('question', '') for qa in qa_dataset]
    embs = embedder.encode(questions, normalize_embeddings=True, show_progress_bar=True)

    for i, qa in enumerate(qa_dataset):
        q_norm = qa.get('question', '').strip().lower()
        hop_type = qa.get('hop_type', '')
        emb = embs[i]

        # 1. Exact string match
        if q_norm in kept_exact:
            continue

        # 2. Semantic + Hop-type duplicate
        is_duplicate = False
        for k_idx, kept_emb in enumerate(kept_embs):
            sim = float(np.dot(emb, kept_emb))
            if sim > threshold and kept_hop_types[k_idx] == hop_type:
                is_duplicate = True
                break

        if not is_duplicate:
            keep.append(qa)
            kept_embs.append(emb)
            kept_hop_types.append(hop_type)
            kept_exact.add(q_norm)

    log.info("Dedup: %d -> %d pairs", len(qa_dataset), len(keep))
    return keep

qa_dataset = dedup_questions(qa_dataset, embedder, threshold=DEDUP_SIM_THRESHOLD)


# ── Save ──
ts = datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(QA_DATASET_DIR, exist_ok=True)

json_path = f"{QA_DATASET_DIR}/tiet_rag_dataset_{ts}.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(qa_dataset, f, indent=2, ensure_ascii=False)

rows = []
for qa in qa_dataset:
    rows.append({
        "question":          qa.get('question', ''),
        "answer":            qa.get('answer', ''),
        "hop_type":          qa.get('hop_type', ''),
        "hop_count":         qa.get('hop_count', 1),
        "question_type":     qa.get('question_type', ''),
        "difficulty":        qa.get('difficulty', ''),
        "grounded":          qa.get('grounded', False),
        "grounded_score":    qa.get('grounded_score', 0.0),
        "is_cross_document": qa.get('is_cross_document', False),
        "is_cross_chunk":    qa.get('is_cross_chunk', False),
        "reasoningchain":    json.dumps(qa.get('reasoning_chain', [])),
        "evidence":          qa.get('evidence', '') or qa.get('matched_evidence', ''),
        "key_facts":         json.dumps(qa.get('key_facts', [])),
        "sourcesections":    " | ".join(qa.get('source_sections', [])),
        "sources":           " | ".join(qa.get('sources', [])),
        "categories":        " | ".join(qa.get('categories', [])),
        "sourcechunks":      json.dumps(qa.get('source_chunks', [])),
    })

df = pd.DataFrame(rows)
csv_path = f"{QA_DATASET_DIR}/tiet_rag_dataset_{ts}.csv"
df.to_csv(csv_path, index=False)

log.info("FINAL DATASET: %d pairs", len(qa_dataset))
log.info("Hop types:\n%s", df['hop_type'].value_counts().to_string())
log.info("Question types:\n%s", df['question_type'].value_counts().to_string())
log.info("Difficulty:\n%s", df['difficulty'].value_counts().to_string())
log.info("Cross-document: %d | Cross-chunk: %d", df['is_cross_document'].sum(), df['is_cross_chunk'].sum())
log.info("Saved: %s", json_path)
log.info("Saved: %s", csv_path)

# Graph summary
print(f"\n{'='*50}")
print("  KNOWLEDGE GRAPH STATISTICS")
print(f"{'='*50}")
print(f"  Total nodes:          {G.number_of_nodes()}")
print(f"  Total edges:          {G.number_of_edges()}")
node_types = {}
for _, d in G.nodes(data=True):
    t = d.get("type", "UNKNOWN")
    node_types[t] = node_types.get(t, 0) + 1
print(f"  Node types:")
for t, c in sorted(node_types.items(), key=lambda x: -x[1]):
    print(f"    {t}: {c}")
edge_relations = {}
for _, _, d in G.edges(data=True):
    r = d.get("relation", "RELATED_TO")
    edge_relations[r] = edge_relations.get(r, 0) + 1
print(f"  Edge relations:")
for r, c in sorted(edge_relations.items(), key=lambda x: -x[1]):
    print(f"    {r}: {c}")
long_dist = sum(1 for _, _, d in G.edges(data=True) if d.get("is_long_distance"))
print(f"  Long-distance edges:  {long_dist}")
print(f"{'='*50}\n")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]


  KNOWLEDGE GRAPH STATISTICS
  Total nodes:          53
  Total edges:          24
  Node types:
    ELIGIBILITY: 11
    CONDITION: 9
    PROCESS: 9
    RULE: 6
    CONSEQUENCE: 5
    SERVICE: 5
    ROLE: 3
    FORM: 2
    COMMITTEE: 1
    PORTAL: 1
    POLICY: 1
  Edge relations:
    REQUIRES: 8
    DEPENDS_ON: 6
    CONTAINS: 5
    EXCEPTION_TO: 2
    APPLIES_TO: 2
    GOVERNED_BY: 1
  Long-distance edges:  0

